# 🚀 ENTRENAMIENTO LLAMA-3.2-3B CON LoRA

---

## 📊 Especificaciones

- **Modelo:** Meta-Llama-3.2-3B-Instruct (3B parámetros)
- **Técnica:** QLoRA (4-bit quantization)
- **GPU:** T4 (15GB VRAM) ✅ PERFECTO
- **Tiempo:** 30-45 minutos ⚡
- **Loss esperado:** 0.3-0.5

---

## ✅ VENTAJAS vs Llama-3-8B

- ✅ **3x más rápido** (30-45 min vs 90-120 min)
- ✅ **Menos RAM** (8GB vs 18GB)
- ✅ **No crashea** en Colab T4
- ✅ **Calidad similar** para español
- ✅ **Más reciente** (Llama 3.2 es 2024)

---

## 📋 ORDEN DE EJECUCIÓN

1. ✅ Paso 1: Instalar dependencias
2. ✅ Paso 2: Verificar GPU
3. ✅ Paso 3: Subir dataset
4. ✅ Paso 4: Token HuggingFace
5. ✅ Paso 5: **ABRIR TENSORBOARD** (antes de entrenar)
6. ✅ Paso 6: Entrenar modelo (30-45 min)
7. ✅ Paso 7: Descargar adaptadores

---

## 📦 PASO 1: INSTALAR DEPENDENCIAS

In [ ]:
%%capture

# Instalar últimas versiones compatibles
!pip install -q --upgrade transformers
!pip install -q --upgrade peft
!pip install -q --upgrade accelerate
!pip install -q --upgrade bitsandbytes
!pip install -q datasets
!pip install -q sentencepiece
!pip install -q einops

print("✅ Dependencias instaladas")

## 🎮 PASO 2: VERIFICAR GPU

In [ ]:
import torch

print("=" * 70)
print("🎮 VERIFICACIÓN DE GPU")
print("=" * 70)

if torch.cuda.is_available():
    print(f"✅ GPU disponible: {torch.cuda.get_device_name(0)}")
    print(f"📊 Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"🚀 CUDA Version: {torch.version.cuda}")
else:
    print("❌ GPU NO disponible")
    print("⚠️  Ve a Runtime → Change runtime type → T4 GPU")

print("=" * 70)

## 📤 PASO 3: SUBIR DATASET

In [ ]:
from google.colab import files
import json

print("📤 Sube tu archivo dataset_pedagogico.json")
print("   (Haz clic en 'Choose Files')\n")

uploaded = files.upload()

if 'dataset_pedagogico.json' in uploaded:
    with open('dataset_pedagogico.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    print(f"\n✅ Dataset cargado: {len(data)} ejemplos")
    print(f"\n📝 Primer ejemplo:")
    print(f"   Instrucción: {data[0]['instruction'][:60]}...")
else:
    print("❌ Error: No se encontró dataset_pedagogico.json")

## 🔑 PASO 4: TOKEN HUGGINGFACE

In [ ]:
from getpass import getpass

print("🔑 TOKEN HUGGINGFACE")
print("=" * 70)
print("1. Obtén token: https://huggingface.co/settings/tokens")
print("2. Acepta licencia: https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct")
print("=" * 70)
print()

HF_TOKEN = getpass("Ingresa tu HuggingFace token: ")

if HF_TOKEN:
    print("\n✅ Token configurado correctamente")
else:
    print("\n❌ Error: Token vacío")

## 📊 PASO 5: ABRIR TENSORBOARD (ANTES DE ENTRENAR)

**⚠️ Ejecuta esta celda ANTES del entrenamiento para ver las gráficas en vivo**

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/llama32

print("\n📊 TensorBoard abierto")
print("   Se actualizará cada 30 segundos")
print("   Verás la curva de loss en tiempo real")

## 🏋️ PASO 6: ENTRENAR LLAMA-3.2-3B

**Tiempo estimado:** 30-45 minutos ⚡

In [ ]:
import json
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training
)

print("=" * 70)
print("🚀 ENTRENAMIENTO LLAMA-3.2-3B CON QLoRA")
print("=" * 70)

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

# Cuantización 4-bit
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "v_proj", "k_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# Training config SIMPLIFICADO (sin torch_compile)
training_args = TrainingArguments(
    output_dir="./lora_model",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_dir="./logs/llama32",
    logging_steps=5,
    save_steps=100,
    save_total_limit=2,
    warmup_steps=30,
    weight_decay=0.01,
    max_grad_norm=1.0,
    optim="paged_adamw_8bit",
    report_to="tensorboard",
    disable_tqdm=False,  # Mostrar barra de progreso
    logging_first_step=True,  # Log inmediato
)

print(f"\n📦 Modelo: {MODEL_NAME}")
print(f"🔧 LoRA: r={lora_config.r}, alpha={lora_config.lora_alpha}")
print(f"🏋️  Épocas: {training_args.num_train_epochs}")
print(f"📊 Batch efectivo: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"📉 Learning rate: {training_args.learning_rate}")
print(f"⏱️  Tiempo estimado: 30-45 minutos")
print(f"📊 TensorBoard: logs/llama32")

# ============================================================================
# PREPARAR DATASET
# ============================================================================

print("\n📚 Preparando dataset...")

with open('dataset_pedagogico.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

def format_instruction(example):
    """Formato Llama-3.2"""
    text = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Eres un asistente educativo experto. Responde SIEMPRE en español con tono pedagógico, motivador y amigable. Usa emojis apropiados.<|eot_id|><|start_header_id|>user<|end_header_id|>

{example['instruction']}

{example['input']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{example['output']}<|eot_id|>"""
    return {"text": text}

dataset = Dataset.from_list(data)
dataset = dataset.map(format_instruction)

print(f"   ✅ {len(dataset)} ejemplos preparados")

# ============================================================================
# CARGAR MODELO
# ============================================================================

print("\n🤖 Cargando Llama-3.2-3B con cuantización 4-bit...")
print("   (Descargando ~6GB, puede tardar 2-3 minutos)")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    token=HF_TOKEN,
    trust_remote_code=True
)

print(f"   ✅ Modelo cargado en GPU (4-bit)")

model = prepare_model_for_kbit_training(model)

# ============================================================================
# APLICAR LoRA
# ============================================================================

print("\n🔧 Aplicando adaptadores LoRA...")

model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
trainable_percentage = 100 * trainable_params / total_params

print(f"\n📊 ESTADÍSTICAS:")
print(f"   Total: {total_params:,} parámetros")
print(f"   Entrenables: {trainable_params:,} ({trainable_percentage:.4f}%)")

# ============================================================================
# TOKENIZAR
# ============================================================================

print("\n📝 Tokenizando dataset...")

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset.column_names
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print(f"   ✅ Dataset tokenizado")

# ============================================================================
# ENTRENAR
# ============================================================================

print("\n" + "=" * 70)
print("🚀 INICIANDO ENTRENAMIENTO")
print("=" * 70)
print(f"⏱️  Tiempo estimado: 30-45 minutos")
print(f"📉 Loss esperado: 1.5 → 0.3-0.5")
print(f"📊 Mira TensorBoard arriba ↑")
print()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

# ENTRENAR (debería empezar inmediatamente)
print("🔥 Empezando entrenamiento...")
trainer.train()

print("\n" + "=" * 70)
print("✅ ENTRENAMIENTO COMPLETADO")
print("=" * 70)

final_loss = trainer.state.log_history[-1].get('loss', 'N/A')
print(f"\n📊 Loss final: {final_loss}")

if isinstance(final_loss, float):
    if final_loss < 0.3:
        print("✅ ¡EXCELENTE! Loss <0.3")
    elif final_loss < 0.5:
        print("✅ MUY BIEN! Loss <0.5")
    elif final_loss < 0.8:
        print("⚠️  ACEPTABLE - Loss <0.8")
    else:
        print("❌ ALTA - Loss >0.8")

# ============================================================================
# GUARDAR
# ============================================================================

print("\n💾 Guardando adaptadores...")

model.save_pretrained("./lora_adapters")
tokenizer.save_pretrained("./lora_adapters")

print(f"   ✅ Guardado en: ./lora_adapters")

# ============================================================================
# PROBAR
# ============================================================================

print("\n🧪 PROBANDO MODELO...")
print("=" * 70)

model.eval()

test_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Eres un asistente educativo experto. Responde SIEMPRE en español con tono pedagógico, motivador y amigable. Usa emojis apropiados.<|eot_id|><|start_header_id|>user<|end_header_id|>

Explica qué es una derivada

Necesito entender el concepto de derivada<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        repetition_penalty=1.2
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("📝 RESPUESTA:")
print("-" * 70)
print(response)
print("-" * 70)

print("\n✅ PROCESO COMPLETADO")

## 📥 PASO 7: DESCARGAR ADAPTADORES

In [ ]:
import shutil
from google.colab import files

print("📦 Comprimiendo adaptadores...")

shutil.make_archive('lora_adapters_llama32_3b', 'zip', './lora_adapters')

print("✅ Adaptadores comprimidos")
print("\n📥 Descargando...")

files.download('lora_adapters_llama32_3b.zip')

print("\n" + "=" * 70)
print("✅ DESCARGA COMPLETADA")
print("=" * 70)
print("\n📋 SIGUIENTES PASOS:")
print("   1. Descomprime lora_adapters_llama32_3b.zip")
print("   2. Renombra a 'lora_adapters'")
print("   3. Copia a: agent-education/fine_tuning/lora_adapters/")
print("   4. Actualiza .env:")
print("      HUGGINGFACE_MODEL=meta-llama/Llama-3.2-3B-Instruct")
print("   5. Reinicia el backend")
print("\n🎉 ¡Llama-3.2-3B listo!")